# WorkToDo – Seven-Factor ML Recommendation Scoring & Ranking
Criteria-based scoring model (synthetic data). The model learns the agreed weighted policy; it does **not** learn real user preferences.
Run all cells top to bottom in a fresh Colab runtime. Upload **both** ZIPs when prompted (datasets ZIP required, ML package ZIP optional cross-check).

## 1. Imports and configuration

In [ ]:
import os, sys, json, time, glob, zipfile, warnings, subprocess, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import joblib, sklearn, scipy
from scipy.stats import spearmanr, loguniform, rankdata
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, ndcg_score
from sklearn.inspection import permutation_importance
try:
    from xgboost import XGBRegressor
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--retries', '1', '--timeout', '20', 'xgboost'], check=False)
    try:
        from xgboost import XGBRegressor
    except ImportError:
        XGBRegressor = None
try:
    from IPython.display import display
except ImportError:
    display = print
warnings.filterwarnings('ignore')

SEED, TEST_SIZE, CV_FOLDS, TUNE_ROWS, TOP_K = 42, 0.2, 4, 20000, 5
WORK_DIR, OUT_DIR = Path('worktodo_data'), Path('worktodo_artifacts')
WORK_DIR.mkdir(exist_ok=True); OUT_DIR.mkdir(exist_ok=True)
np.random.seed(SEED)
print('XGBoost available:', XGBRegressor is not None)

## 2. Upload and extract ZIP files

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
zip_dir = os.environ.get('WORKTODO_ZIP_DIR')
if zip_dir is None:
    if IN_COLAB:
        from google.colab import files
        print('Select BOTH ZIP files (datasets ZIP is required).')
        files.upload()
    zip_dir = '.'
zips = sorted(glob.glob(os.path.join(zip_dir, '*.zip')))
assert zips, 'No ZIP files found - upload the dataset ZIP.'
for z in zips:
    zipfile.ZipFile(z).extractall(WORK_DIR / Path(z).stem)

def find(name, required=True):
    hits = glob.glob(str(WORK_DIR / '**' / name), recursive=True)
    if required: assert hits, f'{name} not found in uploaded ZIPs'
    return hits[0] if hits else None

P = {k: find(v) for k, v in dict(tasks='worktodo_tasks_10000.csv', sessions='worktodo_recommendation_sessions_5000.csv',
                                 cands='worktodo_recommendation_candidates.csv', guide='worktodo_recommendation_schema_guide.csv').items()}
P_FEAT = find('worktodo_training_features.csv', required=False)
P_AUDIT = find('AUDIT_REPORT.md', required=False)
print({k: Path(v).name for k, v in P.items()}, '| provided features:', bool(P_FEAT))

## 3. Reusable feature-engineering module
Written to `worktodo_features.py` so the exact same code is used in training, the demo and the future FastAPI service.

In [ ]:
MODULE_SRC = r'''"""WorkToDo seven-factor feature engineering + ranking helpers (usable outside Colab / in FastAPI)."""
import numpy as np
import pandas as pd

WEIGHTS = {'urgency': 0.30, 'importance': 0.20, 'goal_contribution': 0.15, 'feasibility': 0.15,
           'consequence_of_delay': 0.10, 'timing_suitability': 0.05, 'energy_suitability': 0.05}
FEATURES = list(WEIGHTS)
IMPORTANCE_MAP = {'Critical': 100, 'High': 80, 'Medium': 50, 'Low': 20}
GOAL_MAP = {'Essential': 100, 'High': 75, 'Medium': 50, 'Low': 25, 'No linked goal': 0}
CONSEQUENCE_MAP = {'Major': 100, 'Significant': 75, 'Moderate': 50, 'Minor': 25, 'None': 0, 'No consequence': 0}
LEVEL = {'Low': 0, 'Medium': 1, 'High': 2}            # user energy / focus
REQUIREMENT = {'Simple': 0, 'Moderate': 1, 'Complex': 2}  # task demand
# LEVEL_MATCH[user_level, task_demand] -> 0-100 fit (symmetric; exact match = 100). Constants are inherited from
# the dataset's feature generator (not derived from real behaviour) -> review before production.
LEVEL_MATCH = np.array([[100.0, 67.5, 170 / 3], [67.5, 100.0, 235 / 3], [170 / 3, 235 / 3, 100.0]])
PERIODS = {'Morning', 'Afternoon', 'Evening', 'Night'}
ACTIVE_STATUSES = {'Pending', 'In Progress'}


def _map(s, mapping, name):
    s = pd.Series(s)
    out = s.map(mapping)
    if out.isna().any():
        raise ValueError(f"{name}: unmapped/missing values {sorted(map(str, s[out.isna()].unique()))}")
    return out.astype(float)


def days_until_deadline(due_date, session_timestamp):
    due = pd.to_datetime(pd.Series(due_date)).dt.normalize()
    now = pd.to_datetime(pd.Series(session_timestamp)).dt.normalize()
    return (due - now).dt.days.astype('float64')


def urgency_score(days):
    d = pd.Series(days, dtype='float64')
    conds = [d.isna(), d < 0, d == 0, d == 1, d <= 3, d <= 7, d <= 14]
    return pd.Series(np.select(conds, [10, 100, 95, 85, 70, 50, 30], default=15), index=d.index, dtype=float)


def feasibility_score(duration, min_session, splittable, longest_slot, available=None):
    """100 = whole task fits the longest uninterrupted slot; 60-90 = splittable and a useful first session fits
    (partial progress); 0-40 = cannot meaningfully start (scaled by slot/duration). Total free time is NOT used."""
    d = np.asarray(duration, float); m = np.asarray(min_session, float)
    L = np.asarray(longest_slot, float); s = np.asarray(splittable, bool)
    if available is not None:
        L = np.minimum(L, np.asarray(available, float))
    ratio = np.divide(L, d, out=np.zeros_like(L), where=d > 0)
    return np.where(d <= L, 100.0, np.where(s & (m <= L), 60 + 30 * ratio, 40 * ratio))


def timing_score(preferred, current):
    p = np.asarray(preferred, dtype=object); c = np.asarray(current, dtype=object)
    return np.where(p == 'Anytime', 70.0, np.where(p == c, 100.0, 35.0))


def energy_score(energy, focus, complexity):
    e = _map(energy, LEVEL, 'energy_level').astype(int).to_numpy()
    f = _map(focus, LEVEL, 'focus_level').astype(int).to_numpy()
    r = _map(complexity, REQUIREMENT, 'complexity').astype(int).to_numpy()
    return (LEVEL_MATCH[e, r] + LEVEL_MATCH[f, r]) / 2


def build_features(df):
    """df needs raw task + session columns (see rank_tasks). Returns the seven 0-100 factors."""
    idx = df.index
    days = days_until_deadline(df['due_date'], df['session_timestamp']).set_axis(idx)
    out = pd.DataFrame(index=idx)
    out['urgency'] = urgency_score(days)
    out['importance'] = _map(df['importance'], IMPORTANCE_MAP, 'importance').set_axis(idx)
    out['goal_contribution'] = _map(df['goal_contribution'], GOAL_MAP, 'goal_contribution').set_axis(idx)
    out['feasibility'] = feasibility_score(df['estimated_duration_minutes'], df['minimum_session_minutes'],
                                           df['is_splittable'], df['longest_free_slot_minutes'],
                                           df['available_time_minutes'])
    out['consequence_of_delay'] = _map(df['consequence_of_delay'], CONSEQUENCE_MAP, 'consequence_of_delay').set_axis(idx)
    if not set(df['preferred_time'].unique()) <= PERIODS | {'Anytime'}:
        raise ValueError('unknown preferred_time')
    out['timing_suitability'] = timing_score(df['preferred_time'], df['current_time_period'])
    out['energy_suitability'] = energy_score(df['energy_level'], df['focus_level'], df['complexity'])
    return out[FEATURES]


def weighted_score(F):
    return F[FEATURES].mul(pd.Series(WEIGHTS)).sum(axis=1)


def eligibility(df):
    """Hard rules, independent of ML. Returns exclusion reason ('' = eligible)."""
    today = pd.to_datetime(df['session_timestamp']).dt.normalize()
    start = pd.to_datetime(df['start_date'])
    reason = np.select([~df['status'].isin(ACTIVE_STATUSES), df['dependency_status'].eq('Blocked'), start > today],
                       ['Task not active', 'Blocked dependency', 'Start date in future'], default='')
    return pd.Series(reason, index=df.index)


def predict_scores(model, F):
    return np.clip(model.predict(F[FEATURES]), 0, 100)


def _urgency_text(d):
    if pd.isna(d): return 'no deadline'
    d = int(d)
    return f'overdue by {-d} day(s)' if d < 0 else ('due today' if d == 0 else f'due in {d} day(s)')


def _factor_text(name, r):
    v = r[name]
    if name == 'urgency': return _urgency_text(r['days_until_deadline'])
    if name == 'importance': return f"{r['importance_label']} importance"
    if name == 'goal_contribution': return 'no linked goal' if v == 0 else f"{r['goal_label']} goal contribution"
    if name == 'consequence_of_delay': return f"{r['consequence_label']} consequence if delayed"
    if name == 'feasibility':
        return 'fits the longest free slot' if v == 100 else ('partial progress possible in the free slot' if v >= 60
                                                              else 'does not fit the free slot')
    if name == 'timing_suitability':
        return 'matches preferred time of day' if v == 100 else ('Anytime task' if v == 70
                                                                 else f"preferred time ({r['preferred_time']}) differs from now")
    return f'energy/focus fit {v:.0f}/100'


def assign_categories(ranked, n_now=1, n_next=2):
    """Illustrative deterministic rule (confirm before production): best startable task = Do now, next
    n_next startable tasks = Do next, everything else (incl. high scorers that cannot start now) = Schedule later."""
    ok = ranked['startable_now'].to_numpy(); cs = np.cumsum(ok)
    return np.where(ok & (cs <= n_now), 'Do now', np.where(ok & (cs <= n_now + n_next), 'Do next', 'Schedule later'))


def rank_tasks(tasks, session, model, n_now=1, n_next=2):
    """tasks: DataFrame (task_id, task_name, start_date, due_date, importance, goal_contribution, consequence_of_delay,
    complexity, estimated_duration_minutes, minimum_session_minutes, is_splittable, preferred_time, dependency_status,
    status). session: dict (session_timestamp, available_time_minutes, longest_free_slot_minutes, current_time_period,
    energy_level, focus_level). Returns (ranked, excluded). ML scores; deterministic rules decide eligibility/category."""
    df = tasks.copy().reset_index(drop=True)
    for k, v in session.items():
        df[k] = v
    df['exclusion_reason'] = eligibility(df)
    excluded = df[df['exclusion_reason'] != ''][['task_id', 'exclusion_reason']].reset_index(drop=True)
    df = df[df['exclusion_reason'] == ''].reset_index(drop=True)
    if df.empty:
        return df, excluded
    F = build_features(df)
    df['importance_label'], df['goal_label'], df['consequence_label'] = df['importance'], df['goal_contribution'], df['consequence_of_delay']
    df = pd.concat([df.drop(columns=['importance', 'goal_contribution', 'consequence_of_delay']), F], axis=1)
    df['ml_score'] = predict_scores(model, F)
    df['days_until_deadline'] = days_until_deadline(df['due_date'], df['session_timestamp'])
    df = df.sort_values(['ml_score', 'urgency'], ascending=False, kind='mergesort').reset_index(drop=True)
    df.insert(0, 'rank', np.arange(1, len(df) + 1))
    slot = np.minimum(df['longest_free_slot_minutes'], df['available_time_minutes'])
    df['startable_now'] = df['minimum_session_minutes'] <= slot
    df['category'] = assign_categories(df, n_now, n_next)
    reasons = []
    for _, r in df.iterrows():
        contrib = {k: WEIGHTS[k] * r[k] for k in FEATURES}
        top = sorted(contrib, key=contrib.get, reverse=True)[:3]
        weak = [k for k in FEATURES if r[k] <= 40 and k not in top and WEIGHTS[k] >= 0.05]
        txt = 'Drivers: ' + '; '.join(_factor_text(k, r) for k in top) + '.'
        if weak: txt += ' Held back by: ' + '; '.join(_factor_text(k, r) for k in weak) + '.'
        reasons.append(txt)
    df['reason'] = reasons
    return df, excluded
'''
Path('worktodo_features.py').write_text(MODULE_SRC)
sys.path.insert(0, os.getcwd())
import importlib, worktodo_features as wf
importlib.reload(wf)
FEATURES, WEIGHTS = wf.FEATURES, wf.WEIGHTS
assert abs(sum(WEIGHTS.values()) - 1) < 1e-12

## 4. Inspect datasets and read the audit/schema documentation

In [ ]:
if P_AUDIT: print(Path(P_AUDIT).read_text())
guide = pd.read_csv(P['guide'])
print('\nOutcome / engine-output columns (must never be inputs):')
print(guide.loc[guide.role.str.contains('Outcome|Engine output|Interaction event', na=False), ['dataset', 'column']].to_string(index=False))

tasks, sess, cand = (pd.read_csv(P[k]) for k in ['tasks', 'sessions', 'cands'])
for name, d, pk in [('tasks', tasks, 'task_id'), ('sessions', sess, 'session_id'), ('candidates', cand, 'candidate_id')]:
    nulls = d.isna().sum(); nulls = nulls[nulls > 0].to_dict()
    print(f'\n[{name}] shape={d.shape} full-row duplicates={d.duplicated().sum()} pk_unique={d[pk].is_unique}')
    print('  nulls (semantic unless noted):', nulls)
for name, d, cols in [('tasks', tasks, ['importance', 'goal_contribution', 'consequence_of_delay', 'complexity', 'preferred_time', 'dependency_status', 'task_status_at_baseline', 'is_splittable']),
                      ('sessions', sess, ['current_time_period', 'energy_level', 'focus_level']),
                      ('candidates', cand, ['task_status_at_recommendation', 'eligible_for_recommendation', 'exclusion_reason'])]:
    for c in cols: print(f'  {name}.{c}:', d[c].value_counts(dropna=False).to_dict())

## 5. Validate relationships

In [ ]:
def check(label, ok):
    print(('PASS ' if ok else 'FAIL ') + label); assert ok, label

check('primary keys unique', tasks.task_id.is_unique and sess.session_id.is_unique and cand.candidate_id.is_unique)
check('candidates.session_id -> sessions', cand.session_id.isin(sess.session_id).all())
check('candidates.task_id -> tasks', cand.task_id.isin(tasks.task_id).all())
check('no duplicate (session_id, task_id) pairs', not cand.duplicated(['session_id', 'task_id']).any())
df = (cand.merge(sess.drop(columns=['user_id']), on='session_id', how='left', validate='m:1')
          .merge(tasks.drop(columns=['user_id']), on='task_id', how='left', validate='m:1'))
check('user_id consistent across candidates/sessions/tasks',
      (cand.merge(sess[['session_id', 'user_id']], on='session_id', suffixes=('', '_s')).eval('user_id == user_id_s').all()) and
      (cand.merge(tasks[['task_id', 'user_id']], on='task_id', suffixes=('', '_t')).eval('user_id == user_id_t').all()))
check('longest_free_slot <= available_time', (sess.longest_free_slot_minutes <= sess.available_time_minutes).all())
check('split-task minimum <= duration', (tasks.minimum_session_minutes <= tasks.estimated_duration_minutes).all())
check('non-splittable minimum == duration', (tasks.loc[~tasks.is_splittable, 'minimum_session_minutes'] == tasks.loc[~tasks.is_splittable, 'estimated_duration_minutes']).all())
print(f"users={cand.user_id.nunique()}, candidates/session min-max={cand.groupby('session_id').size().min()}-{cand.groupby('session_id').size().max()}")

# Hard eligibility rebuilt from raw columns and compared with the dataset's own flag
df['status'] = df['task_status_at_recommendation']
df['exclusion_reason_rebuilt'] = wf.eligibility(df)
check('rebuilt eligibility == dataset eligible_for_recommendation', ((df.exclusion_reason_rebuilt == '') == (df.eligible_for_recommendation == 1)).all())
print('Ineligible rows kept in the data but excluded from ML training/ranking by hard rules:', df.exclusion_reason_rebuilt.value_counts().drop('', errors='ignore').to_dict())

## 6. Feature engineering and target
Seven factors rebuilt from the raw linked tables (feasibility uses `longest_free_slot_minutes`, `is_splittable`, `minimum_session_minutes`; not total calendar time). Target = agreed weighted formula.

In [ ]:
F = wf.build_features(df)
chk_days = wf.days_until_deadline(df.due_date, df.session_timestamp)
check('rebuilt days_until_deadline == dataset column', np.allclose(chk_days.fillna(-999), df.days_until_deadline_at_recommendation.fillna(-999)))
check('all factors within 0-100', F.min().min() >= 0 and F.max().max() <= 100)
raw_df = df.copy()   # raw categorical columns kept for the ranking demo
df[FEATURES] = F
df['target_score'] = wf.weighted_score(F)

if P_FEAT:  # cross-check against the provided feature-ready CSV
    prov = pd.read_csv(P_FEAT).rename(columns={'importance_factor': 'importance', 'goal_factor': 'goal_contribution', 'consequence_factor': 'consequence_of_delay'})
    m = df.merge(prov, on='candidate_id', suffixes=('', '_prov'), validate='1:1')
    diffs = {c: float(np.abs(m[c] - m[c + '_prov']).max()) for c in FEATURES}
    diffs['target'] = float(np.abs(m.target_score - m.target_recommendation_score).max())
    print(f'Rebuilt vs provided features on {len(m)} rows - max abs diff:', {k: round(v, 9) for k, v in diffs.items()})
    check('rebuilt features/target reproduce provided CSV', max(diffs.values()) < 1e-6)

data = df[df.exclusion_reason_rebuilt == ''].reset_index(drop=True)
print('\nModelling rows:', len(data), '| sessions:', data.session_id.nunique(), '| users:', data.user_id.nunique())
display(data[FEATURES + ['target_score']].describe().T[['mean', 'std', 'min', 'max']].round(2))
print('Max |corr| between factors:', data[FEATURES].corr().abs().where(lambda c: c < 1).max().max().round(3))

## 7. Leakage guard and user-grouped 80:20 split

In [ ]:
OUTCOME_COLS = ['accepted', 'dismissed', 'ignored', 'started', 'completed_after_recommendation', 'actual_duration_minutes_after_recommendation',
                'completed_at_after_recommendation', 'completed_on_time_after_recommendation', 'recommendation_score', 'recommendation_category',
                'recommendation_reason', 'final_recommendation_position', 'user_action', 'label_source']
check('no outcome/engine-output column used as feature', set(FEATURES).isdisjoint(OUTCOME_COLS))
check('no ID column used as feature', not any(c.endswith('_id') for c in FEATURES))

gss = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
tr_idx, te_idx = next(gss.split(data, groups=data.user_id))
train, test = data.iloc[tr_idx].copy(), data.iloc[te_idx].copy()
check('no user in both train and test', set(train.user_id).isdisjoint(test.user_id))
check('no session in both train and test', set(train.session_id).isdisjoint(test.session_id))
print(f'train rows={len(train)} users={train.user_id.nunique()} sessions={train.session_id.nunique()} | test rows={len(test)} users={test.user_id.nunique()} sessions={test.session_id.nunique()}')
X_train, y_train, X_test, y_test = train[FEATURES], train.target_score, test[FEATURES], test.target_score

## 8. Models, preprocessing and grouped hyperparameter tuning
Tuning uses grouped CV (by `user_id`) on a training subsample only; the best parameters are then refit on the full training set. The test set is untouched.

In [ ]:
def make_pipe(est, scale):
    prep = ColumnTransformer([('num', StandardScaler() if scale else 'passthrough', FEATURES)], remainder='drop')
    return Pipeline([('prep', prep), ('model', est)])

MODELS = {
    'Linear Regression': dict(est=LinearRegression(), scale=True, params=None, n_iter=0, cx=1),
    'Ridge Regression': dict(est=Ridge(random_state=SEED), scale=True, params={'alpha': loguniform(1e-3, 10)}, n_iter=6, cx=2),
    'Random Forest': dict(est=RandomForestRegressor(random_state=SEED, n_jobs=1), scale=False, cx=4, n_iter=6,
                          params={'n_estimators': [100, 200], 'max_depth': [8, 12, 16, 24], 'min_samples_leaf': [1, 3, 5], 'max_features': [0.6, 1.0]}),
    'HistGradientBoosting': dict(est=HistGradientBoostingRegressor(random_state=SEED, early_stopping=False), scale=False, cx=3, n_iter=8,
                                 params={'learning_rate': loguniform(0.03, 0.3), 'max_iter': [200, 400], 'max_leaf_nodes': [15, 31, 63], 'min_samples_leaf': [20, 50], 'l2_regularization': [0.0, 1.0]}),
}
if XGBRegressor is not None:
    MODELS['XGBoost'] = dict(est=XGBRegressor(random_state=SEED, tree_method='hist', n_jobs=1, objective='reg:squarederror'), scale=False, cx=5, n_iter=8,
                             params={'n_estimators': [200, 400, 600], 'max_depth': [3, 4, 6], 'learning_rate': loguniform(0.03, 0.3), 'subsample': [0.8, 1.0], 'min_child_weight': [1, 5]})
else:
    print('XGBoost not installed -> skipped (it is preinstalled in Colab).')

tune = X_train.sample(min(TUNE_ROWS, len(X_train)), random_state=SEED)
fitted, best_params, fit_info = {}, {}, {}
for name, spec in MODELS.items():
    t0 = time.perf_counter()
    pipe = make_pipe(spec['est'], spec['scale']); cv_mae = np.nan
    if spec['params']:
        rs = RandomizedSearchCV(pipe, {f'model__{k}': v for k, v in spec['params'].items()}, n_iter=spec['n_iter'], cv=GroupKFold(n_splits=CV_FOLDS),
                                scoring='neg_mean_absolute_error', random_state=SEED, n_jobs=-1, refit=False)
        rs.fit(tune, y_train.loc[tune.index], groups=train.user_id.loc[tune.index])
        pipe.set_params(**rs.best_params_); cv_mae = -rs.best_score_
        best_params[name] = {k.replace('model__', ''): (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in rs.best_params_.items()}
    if 'n_jobs' in pipe.named_steps['model'].get_params(): pipe.set_params(model__n_jobs=-1)
    pipe.fit(X_train, y_train)
    fitted[name] = pipe; fit_info[name] = dict(cv_mae_grouped=cv_mae, fit_seconds=round(time.perf_counter() - t0, 1))
    print(f'{name:22s} grouped-CV MAE={cv_mae:.4f} time={fit_info[name]["fit_seconds"]}s params={best_params.get(name, {})}')

## 9. Evaluation (regression + per-session ranking metrics)
**Relevance for NDCG@5** (computed inside each session, then averaged): within a session, the task ranks 1-5 by *target* score receive graded relevance 5,4,3,2,1 (tied targets share the same grade) and all other tasks 0; predicted scores define the ranking being judged. `NDCG@5 (score-gain)` is a looser variant using the raw 0-100 target as relevance. **Spearman** is the mean per-session rank correlation (pooled value also reported).

In [ ]:
def session_ranking_metrics(sdf, pred, k=TOP_K):
    tmp = pd.DataFrame({'sid': sdf.session_id.to_numpy(), 'y': sdf.target_score.to_numpy(), 'p': np.asarray(pred)})
    sp, ndr, nds = [], [], []
    for _, g in tmp.groupby('sid', sort=False):
        if len(g) < 2: continue
        y, p = g.y.to_numpy(), np.round(g.p.to_numpy(), 6)   # round away float noise so it cannot reorder exact ties
        rel = np.clip(k + 1 - rankdata(-y, method='min'), 0, None)   # tied targets share a grade
        ndr.append(ndcg_score([rel], [p], k=k)); nds.append(ndcg_score([y], [p], k=k))
        sp.append(spearmanr(y, p)[0] if np.ptp(p) > 1e-12 else np.nan)
    return np.nanmean(sp), np.mean(ndr), np.mean(nds)

def evaluate(name, pred):
    sp, ndr, nds = session_ranking_metrics(test, pred)
    return {'Model': name, 'MAE': mean_absolute_error(y_test, pred), 'RMSE': float(np.sqrt(mean_squared_error(y_test, pred))), 'R2': r2_score(y_test, pred),
            'Spearman': sp, 'NDCG@5': ndr, 'NDCG@5 (score-gain)': nds, 'Spearman (pooled)': spearmanr(y_test, pred)[0]}

preds = {n: wf.predict_scores(p, X_test) for n, p in fitted.items()}
baseline_pred = wf.weighted_score(X_test).to_numpy()
rows = [evaluate(n, p) for n, p in preds.items()]
for r in rows: r.update(fit_info[r['Model']]); r['best_params'] = json.dumps(best_params.get(r['Model'], {}), default=str)
comp = pd.DataFrame(rows)

# Model selection: average rank over the 5 metrics (ties -> simpler model). Baseline formula is a reference, not a candidate.
metric_dir = {'MAE': True, 'RMSE': True, 'R2': False, 'Spearman': False, 'NDCG@5': False}
rk = pd.DataFrame({m: comp[m].round(6).rank(ascending=asc, method='min') for m, asc in metric_dir.items()})
comp['avg_rank'] = rk.mean(axis=1)
comp['complexity'] = comp.Model.map({n: s['cx'] for n, s in MODELS.items()})
comp = comp.sort_values(['avg_rank', 'complexity', 'MAE']).reset_index(drop=True)
BEST = comp.loc[0, 'Model']
base_row = evaluate('Weighted formula (baseline)', baseline_pred)
display_cols = ['Model', 'MAE', 'RMSE', 'R2', 'Spearman', 'NDCG@5']
table = pd.concat([comp[display_cols], pd.DataFrame([base_row])[display_cols]], ignore_index=True)
print(table.to_string(index=False, float_format=lambda v: f'{v:.6g}'))
print(f'\nSelected model: {BEST}  (best average rank across MAE, RMSE, R2, Spearman, NDCG@5; ties broken by simplicity)')
print('NOTE: the target is an exact linear function of the 7 inputs, so Linear/Ridge recover it almost perfectly by construction, and the formula baseline is exact (error 0).\n'
      'This verifies the pipeline; it is not evidence of real-world recommendation quality.')

## 10. Visualisations

In [ ]:
best_pipe, best_pred = fitted[BEST], preds[BEST]
FIG = OUT_DIR / 'figures'; FIG.mkdir(exist_ok=True)

# 1. Actual vs predicted
s = np.random.RandomState(SEED).choice(len(test), min(4000, len(test)), replace=False)
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.scatter(y_test.to_numpy()[s], best_pred[s], s=6, alpha=.35); ax.plot([0, 100], [0, 100], 'r--', lw=1, label='Perfect prediction')
ax.set(xlabel='Target score (weighted formula)', ylabel='Predicted score', title=f'Actual vs predicted - {BEST} (test users)'); ax.legend()
fig.tight_layout(); fig.savefig(FIG / '1_actual_vs_predicted.png', dpi=150); plt.show()

# 2. Residual distributions (all models; differences show where approximation error comes from)
names = list(preds); nc = 3; nr = int(np.ceil(len(names) / nc))
fig, axs = plt.subplots(nr, nc, figsize=(12, 3.2 * nr)); axs = np.atleast_1d(axs).ravel()
for ax, n in zip(axs, names):
    res = y_test.to_numpy() - preds[n]; ax.hist(res, bins=50, color='tab:blue'); ax.axvline(0, color='r', lw=1)
    ax.set(title=f'{n}\nmean={res.mean():.2g}, std={res.std():.2g}', xlabel='Residual (target - predicted)', ylabel='Rows')
for ax in axs[len(names):]: ax.axis('off')
fig.suptitle('Residual distribution on test users'); fig.tight_layout(); fig.savefig(FIG / '2_residuals.png', dpi=150); plt.show()

# 3. Model performance comparison (log MAE; NDCG@5)
fig, axs = plt.subplots(1, 2, figsize=(11, 4)); order = comp.Model.tolist()
axs[0].barh(order, comp.MAE.clip(lower=1e-12)); axs[0].set_xscale('log'); axs[0].invert_yaxis(); axs[0].set(title='MAE (lower is better, log scale; floor 1e-12)', xlabel='MAE')
axs[1].barh(order, comp['NDCG@5']); axs[1].invert_yaxis(); axs[1].set(title='NDCG@5 per session (higher is better)', xlabel='NDCG@5', xlim=(max(0, comp['NDCG@5'].min() - 0.02), 1.0))
fig.tight_layout(); fig.savefig(FIG / '3_model_comparison.png', dpi=150); plt.show()

# 4. Feature importance (permutation importance on test subset, all models) + formula weights
sub = X_test.sample(min(5000, len(X_test)), random_state=SEED); ysub = y_test.loc[sub.index]
imp_rows = []
for n, p in fitted.items():
    pi = permutation_importance(p, sub, ysub, scoring='neg_mean_absolute_error', n_repeats=5, random_state=SEED, n_jobs=1)
    mdl, prep = p.named_steps['model'], p.named_steps['prep']
    if hasattr(mdl, 'coef_'): native = mdl.coef_ / prep.named_transformers_['num'].scale_
    elif hasattr(mdl, 'feature_importances_'): native = mdl.feature_importances_
    else: native = [np.nan] * len(FEATURES)
    for i, f in enumerate(FEATURES):
        imp_rows.append({'model': n, 'feature': f, 'permutation_importance_mae_increase': pi.importances_mean[i], 'permutation_std': pi.importances_std[i],
                         'native_coefficient_or_importance': native[i], 'formula_weight': WEIGHTS[f]})
imp = pd.DataFrame(imp_rows)
bi = imp[imp.model == BEST].set_index('feature').loc[FEATURES]
fig, ax = plt.subplots(figsize=(8, 4)); x = np.arange(len(FEATURES)); w = .38
ax.bar(x - w / 2, bi.permutation_importance_mae_increase / bi.permutation_importance_mae_increase.sum(), w, label=f'{BEST}: share of permutation importance')
ax.bar(x + w / 2, bi.formula_weight, w, label='Formula weight')
ax.set_xticks(x); ax.set_xticklabels([f.replace('_', '\n') for f in FEATURES], fontsize=8); ax.set(ylabel='Share', title='Feature importance vs formula weights'); ax.legend(fontsize=8)
fig.text(.01, -.02, 'Permutation importance mixes weight with factor spread, so shares need not equal weights exactly.', fontsize=7)
fig.tight_layout(); fig.savefig(FIG / '4_feature_importance.png', dpi=150, bbox_inches='tight'); plt.show()
if hasattr(best_pipe.named_steps['model'], 'coef_'):
    print('Original-scale coefficients vs formula weights:', {f: (round(float(c), 4), WEIGHTS[f]) for f, c in zip(FEATURES, bi.native_coefficient_or_importance)})

## 11. Recommendation ranking demonstration
ML scores eligible tasks only. Eligibility (completed/cancelled/blocked/future-start) and the Do now / Do next / Schedule later categories are deterministic rules outside the model.

In [ ]:
TASK_COLS = ['task_id', 'task_name', 'start_date', 'due_date', 'importance', 'goal_contribution', 'consequence_of_delay', 'complexity',
             'estimated_duration_minutes', 'minimum_session_minutes', 'is_splittable', 'preferred_time', 'dependency_status']
SESS_COLS = ['session_timestamp', 'available_time_minutes', 'longest_free_slot_minutes', 'current_time_period', 'energy_level', 'focus_level']

def demo_session(session_id, model):
    c = raw_df[raw_df.session_id == session_id]              # all candidates incl. blocked ones
    t = c[TASK_COLS].assign(status=c.task_status_at_recommendation.to_numpy())
    return wf.rank_tasks(t, c.iloc[0][SESS_COLS].to_dict(), model)

test_sessions = sorted(test.session_id.unique())
demo_ids = list(np.random.RandomState(7).choice(test_sessions, 5, replace=False))
cache = test.assign(pred=best_pred).set_index('candidate_id')
FCOLS = ['urgency', 'importance', 'goal_contribution', 'feasibility', 'consequence_of_delay', 'timing_suitability', 'energy_suitability']
examples = []
for i, sid in enumerate(demo_ids):
    ranked, excluded = demo_session(sid, best_pipe)
    ref = cache[cache.session_id == sid].set_index('task_id')['pred'].loc[ranked.task_id.values].to_numpy()
    check(f'{sid}: demo scores identical to evaluation predictions', np.allclose(ranked.ml_score.to_numpy(), ref))
    top = ranked.head(TOP_K).copy(); top.insert(0, 'session_id', sid); examples.append(top)
    if i < 2:
        s0 = ranked.iloc[0]
        print(f"\n=== Session {sid} | {s0.current_time_period} | available {s0.available_time_minutes} min (longest slot {s0.longest_free_slot_minutes}) | energy {s0.energy_level}, focus {s0.focus_level}")
        print(f'Eligible: {len(ranked)} | excluded by hard rules: {excluded.exclusion_reason.value_counts().to_dict()}')
        display(top[['rank', 'task_name', 'ml_score', 'category'] + FCOLS].round(1))
        for _, r in top.iterrows(): print(f"  #{r['rank']} {r.task_name}: {r.reason}")
ex_df = pd.concat(examples, ignore_index=True)

# 5. Example top-5 breakdown (weighted points per factor)
first = ex_df[ex_df.session_id == demo_ids[0]]
fig, ax = plt.subplots(figsize=(9, 4)); left = np.zeros(len(first))
for f in FCOLS:
    pts = (first[f] * WEIGHTS[f]).to_numpy(); ax.barh(first.task_name.str[:32] + ' (#' + first['rank'].astype(str) + ')', pts, left=left, label=f.replace('_', ' ')); left += pts
ax.invert_yaxis(); ax.set(xlabel='Weighted points (sum = score)', title=f'Top-5 recommendations - session {demo_ids[0]} ({BEST})'); ax.legend(fontsize=7, ncol=2)
fig.tight_layout(); fig.savefig(FIG / '5_top5_example.png', dpi=150); plt.show()

## 12. Save and download artifacts

In [ ]:
import joblib as jl
jl.dump(best_pipe, OUT_DIR / 'worktodo_recommendation_model.joblib')
jl.dump(best_pipe.named_steps['prep'], OUT_DIR / 'worktodo_preprocessor.joblib')
comp_out = pd.concat([comp.drop(columns=['complexity']), pd.DataFrame([{**base_row, 'avg_rank': np.nan}])], ignore_index=True)
comp_out.to_csv(OUT_DIR / 'worktodo_model_comparison.csv', index=False)
pred_out = test[['candidate_id', 'session_id', 'user_id', 'task_id'] + FEATURES + ['target_score']].copy()
pred_out['formula_baseline'] = baseline_pred
for n, p in preds.items(): pred_out['pred_' + n.lower().replace(' ', '_')] = p
pred_out['selected_model_prediction'] = best_pred; pred_out['selected_model_residual'] = pred_out.target_score - best_pred
pred_out.to_csv(OUT_DIR / 'worktodo_test_predictions.csv', index=False)
imp.to_csv(OUT_DIR / 'worktodo_feature_importance.csv', index=False)
keep = ['session_id', 'rank', 'task_id', 'task_name', 'ml_score', 'category'] + FCOLS + ['reason']
ex_df[keep].to_csv(OUT_DIR / 'worktodo_example_recommendations.csv', index=False)
(OUT_DIR / 'worktodo_example_recommendations.json').write_text(json.dumps(json.loads(ex_df[keep].to_json(orient='records')), indent=2))
Path(OUT_DIR / 'worktodo_features.py').write_text(MODULE_SRC)

FASTAPI_SRC = '''"""Run: uvicorn fastapi_example_main:app --reload   (needs fastapi, uvicorn, pydantic>=2, scikit-learn, joblib, pandas; xgboost if selected)"""
from pathlib import Path
from typing import List, Optional
import joblib, pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel
import worktodo_features as wf

class Task(BaseModel):
    task_id: str; task_name: str = ''; start_date: str; due_date: Optional[str] = None
    importance: str; goal_contribution: str; consequence_of_delay: str; complexity: str
    estimated_duration_minutes: int; minimum_session_minutes: int; is_splittable: bool
    preferred_time: str; dependency_status: str; status: str

class Session(BaseModel):
    session_timestamp: str; available_time_minutes: int; longest_free_slot_minutes: int
    current_time_period: str; energy_level: str; focus_level: str

class RankRequest(BaseModel):
    session: Session; tasks: List[Task]; top_k: int = 5

app = FastAPI(title="WorkToDo ML scoring service")
model = joblib.load(Path(__file__).with_name("worktodo_recommendation_model.joblib"))  # load once at startup

@app.post("/rank")
def rank(req: RankRequest):
    ranked, excluded = wf.rank_tasks(pd.DataFrame([t.model_dump() for t in req.tasks]), req.session.model_dump(), model)
    cols = ["rank", "task_id", "task_name", "ml_score", "category", "reason"] + wf.FEATURES
    return {"ranked": ranked.head(req.top_k)[cols].to_dict("records") if len(ranked) else [], "excluded": excluded.to_dict("records")}
'''
Path(OUT_DIR / 'fastapi_example_main.py').write_text(FASTAPI_SRC)

meta = dict(
    created_utc=pd.Timestamp.now('UTC').isoformat(), seed=SEED, selected_model=BEST, selection_rule='lowest average rank over MAE, RMSE, R2, Spearman, NDCG@5; ties -> simpler model',
    features=FEATURES, weights=WEIGHTS, target='weighted sum of seven 0-100 factors (agreed formula, weights unchanged)',
    split=dict(method='GroupShuffleSplit by user_id', test_size=TEST_SIZE, train_rows=len(train), test_rows=len(test), train_users=int(train.user_id.nunique()), test_users=int(test.user_id.nunique()),
               train_sessions=int(train.session_id.nunique()), test_sessions=int(test.session_id.nunique())),
    tuning=dict(method='RandomizedSearchCV', cv=f'GroupKFold({CV_FOLDS}) by user_id', scoring='neg_mean_absolute_error', tune_rows=min(TUNE_ROWS, len(train)), best_params=best_params),
    data=dict(candidate_rows=len(df), eligible_rows_used=len(data), excluded_by_hard_rules=int(len(df) - len(data)), prediction_clipped_to=[0, 100]),
    ndcg_definition='per-session NDCG@5; relevance 5,4,3,2,1 for target ranks 1-5 within the session (ties share a grade), 0 otherwise; predictions rounded to 1e-6',
    mappings=dict(importance=wf.IMPORTANCE_MAP, goal=wf.GOAL_MAP, consequence=wf.CONSEQUENCE_MAP),
    versions=dict(python=platform.python_version(), sklearn=sklearn.__version__, pandas=pd.__version__, numpy=np.__version__, scipy=scipy.__version__,
                  xgboost=(sys.modules['xgboost'].__version__ if 'xgboost' in sys.modules else None)),
    limitations=['Synthetic data; target is the agreed formula, so models approximate a known policy.', 'Not validated on real user behaviour; not a personalised model.',
                 'Feasibility split fields and longest_free_slot are synthetic placeholders.', 'Energy match matrix and category rules are inherited/illustrative and need review.'])
(OUT_DIR / 'worktodo_training_config.json').write_text(json.dumps(meta, indent=2, default=str))

try:
    md_table = table.to_markdown(index=False, floatfmt='.6g')
except ImportError:
    md_table = '```\n' + table.to_string(index=False, float_format=lambda v: f'{v:.6g}') + '\n```'
report = f"""# WorkToDo Seven-Factor Scoring Model - Evaluation Report

Selected model: **{BEST}** (test set: {len(test)} rows, {test.user_id.nunique()} users never seen in training).

## Test-set comparison
{md_table}

NDCG@5 and Spearman are computed per session and averaged. NDCG@5 relevance: target ranks 1-5 in a session get 5,4,3,2,1 (ties share a grade); others 0.

## Interpretation
- The target is the agreed weighted formula (an exact linear function of the 7 factors). Linear/Ridge therefore recover it almost perfectly and the formula baseline is exact; tree models only approximate it.
- Selection used the average rank across all five metrics, not R2 alone.
- The model can be used as a scoring/ranking component; eligibility and Do now / Do next / Schedule later categories are deterministic rules outside the model.

## Limitations
""" + '\n'.join('- ' + l for l in meta['limitations']) + """
- Synthetic acceptance/completion labels were not used as targets or features.
"""
(OUT_DIR / 'worktodo_evaluation_report.md').write_text(report)

import shutil
zip_path = shutil.make_archive('worktodo_artifacts', 'zip', OUT_DIR)
print('Saved:', *sorted(p.name for p in OUT_DIR.iterdir()), sep='\n  ')
check('saved model reloads and reproduces test predictions', np.allclose(wf.predict_scores(jl.load(OUT_DIR / 'worktodo_recommendation_model.joblib'), X_test), best_pred))
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)

## 13. Serving through FastAPI (summary)
1. Copy `worktodo_recommendation_model.joblib`, `worktodo_features.py` and `fastapi_example_main.py` into the Python ML service (same scikit-learn version as training; install xgboost if it was selected).
2. Load the model once at startup. The Node/Express backend POSTs the user's active tasks (from PostgreSQL) plus session context (time, energy, focus, free slots) to `/rank`.
3. `rank_tasks` applies hard eligibility rules, computes the seven factors, calls the model, ranks, assigns categories and returns reasons; Node stores/returns the result to the React frontend.
4. Keep eligibility, scheduling constraints and categories in deterministic code; use the model only for scoring. Collect real user feedback now so personalised models can be trained later.